# Uber Ride Analysis and Cancellation Prediction

Este notebook apresenta um fluxo completo de análise exploratória de dados e modelagem inicial para previsão de cancelamento de corridas em uma plataforma de transporte por aplicativo.

## Etapas
- Carregamento e inspeção dos dados
- Limpeza e padronização
- Engenharia de atributos
- Análise exploratória
- Preparação para Machine Learning
- Treinamento e avaliação de modelo


In [ ]:
# Bibliotecas
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid')


## 1. Carregamento dos dados

Ajuste o nome do arquivo abaixo conforme o nome do seu dataset.

In [ ]:
file_path = 'ncr_ride_bookings.csv'  
df = pd.read_csv(file_path)

df.head()


## 2. Entendimento inicial dos dados

In [ ]:
print('Dimensão do dataset:', df.shape)
print('\nInformações gerais:')
df.info()

print('\nValores ausentes por coluna:')
display(df.isnull().sum())

print('\nResumo estatístico:')
display(df.describe(include='all'))


## 3. Padronização dos nomes das colunas

Este passo ajuda a evitar erros no código.

In [ ]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_')
    .str.replace('-', '_')
)

print(df.columns.tolist())


## 4. Ajuste opcional de nomes de colunas

Se o seu dataset tiver nomes um pouco diferentes, ajuste neste dicionário.
Exemplo: `booking_status`, `ride_value`, `driver_ratings`, `customer_rating`, `booking_value`, `date`, `time`, etc.

In [ ]:
# Ajuste apenas se necessário
rename_map = {
    # 'booking_value': 'ride_value',
    # 'customer_ratings': 'customer_rating',
    # 'driver_ratings': 'driver_rating',
}

df = df.rename(columns=rename_map)
print(df.columns.tolist())


## 5. Limpeza dos dados

In [ ]:
# Remover duplicados
df = df.drop_duplicates().copy()

# Preencher colunas numéricas ausentes com a mediana
numeric_cols = df.select_dtypes(include=['number']).columns
for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())

# Preencher colunas categóricas ausentes com 'Unknown'
categorical_cols = df.select_dtypes(include=['object']).columns
for col in categorical_cols:
    df[col] = df[col].fillna('Unknown')

print('Dimensão após limpeza:', df.shape)


## 6. Engenharia de atributos

Esta etapa cria variáveis úteis para análise e modelagem.

In [ ]:
# Criar variável de cancelamento
# Completed = 0 | Demais status = 1
if 'booking_status' in df.columns:
    df['cancelamento'] = np.where(df['booking_status'].astype(str).str.lower() == 'completed', 0, 1)
else:
    raise ValueError("A coluna 'booking_status' não foi encontrada. Ajuste o nome da coluna.")

# Tentar criar variáveis temporais
if 'date' in df.columns:
    df['date'] = pd.to_datetime(df['date'], errors='coerce')
    df['dia_semana'] = df['date'].dt.day_name()
    df['mes'] = df['date'].dt.month

if 'time' in df.columns:
    df['time'] = pd.to_datetime(df['time'], errors='coerce')
    df['hora'] = df['time'].dt.hour
elif 'date' in df.columns:
    df['hora'] = df['date'].dt.hour

if 'hora' in df.columns:
    df['horario_pico'] = np.where(df['hora'].between(7, 9) | df['hora'].between(17, 20), 1, 0)
    df['is_weekend'] = np.where(df['dia_semana'].isin(['Saturday', 'Sunday']), 1, 0) if 'dia_semana' in df.columns else 0

# Criar valor por quilômetro, se houver colunas adequadas
if 'booking_value' in df.columns and 'ride_distance' in df.columns:
    df_valor = df[df['booking_value'] > 0].copy()
    df_valor['valor_por_km'] = df_valor['booking_value'] / (df_valor['ride_distance'] + 0.01)
elif 'ride_value' in df.columns and 'distance' in df.columns:
    df_valor = df[df['ride_value'] > 0].copy()
    df_valor['valor_por_km'] = df_valor['ride_value'] / (df_valor['distance'] + 0.01)
else:
    df_valor = df.copy()

df.head()


## 7. Análise exploratória dos dados (EDA)

In [ ]:
# Distribuição do status da corrida
plt.figure(figsize=(8, 4))
sns.countplot(data=df, x='booking_status', order=df['booking_status'].value_counts().index)
plt.title('Distribuição do status das corridas')
plt.xticks(rotation=45)
plt.show()


In [ ]:
# Cancelamento por horário, se existir a coluna 'hora'
if 'hora' in df.columns:
    plt.figure(figsize=(10, 4))
    sns.countplot(data=df, x='hora', hue='cancelamento')
    plt.title('Cancelamentos por hora do dia')
    plt.show()


In [ ]:
# Forma de pagamento mais utilizada
payment_col = None
for col in ['payment_method', 'payment_type']:
    if col in df.columns:
        payment_col = col
        break

if payment_col:
    plt.figure(figsize=(8, 4))
    sns.countplot(data=df, x=payment_col, order=df[payment_col].value_counts().index)
    plt.title('Distribuição das formas de pagamento')
    plt.xticks(rotation=45)
    plt.show()


In [ ]:
# Distribuição do valor da corrida
value_col = None
for col in ['booking_value', 'ride_value']:
    if col in df_valor.columns:
        value_col = col
        break

if value_col:
    plt.figure(figsize=(8, 4))
    sns.histplot(df_valor[value_col], bins=30, kde=True)
    plt.title('Distribuição do valor das corridas')
    plt.show()


## 8. Preparação para Machine Learning

Nesta etapa, vamos selecionar algumas variáveis e treinar um modelo inicial para previsão de cancelamento.

In [ ]:
# Seleção de colunas candidatas
candidate_features = []
for col in ['hora', 'horario_pico', 'ride_distance', 'distance', 'booking_value', 'ride_value', 'valor_por_km', 'customer_rating', 'driver_rating', 'customer_ratings', 'driver_ratings', 'is_weekend']:
    if col in df_valor.columns:
        candidate_features.append(col)
    elif col in df.columns:
        candidate_features.append(col)

candidate_features = list(dict.fromkeys(candidate_features))
print('Features candidatas:', candidate_features)


In [ ]:
# Base para modelagem
model_df = df.copy()

# Codificar colunas categóricas simples, se existirem
label_cols = []
for col in ['vehicle_type', 'payment_method', 'payment_type', 'pickup_location']:
    if col in model_df.columns:
        label_cols.append(col)

le_dict = {}
for col in label_cols:
    le = LabelEncoder()
    model_df[col] = le.fit_transform(model_df[col].astype(str))
    le_dict[col] = le

for col in label_cols:
    if col not in candidate_features:
        candidate_features.append(col)

X = model_df[candidate_features].copy()
y = model_df['cancelamento'].copy()

print('Formato de X:', X.shape)
print('Formato de y:', y.shape)


## 9. Divisão treino e teste

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('Treino:', X_train.shape)
print('Teste:', X_test.shape)


## 10. Treinamento do modelo

In [ ]:
model = RandomForestClassifier(random_state=42, n_estimators=200, max_depth=8)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)


## 11. Avaliação do modelo

In [ ]:
print('Accuracy:', round(accuracy_score(y_test, y_pred), 4))
print('\nMatriz de confusão:')
print(confusion_matrix(y_test, y_pred))
print('\nRelatório de classificação:')
print(classification_report(y_test, y_pred))


In [ ]:
# Importância das variáveis
importances = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=True)

plt.figure(figsize=(8, 5))
importances.plot(kind='barh')
plt.title('Importância das variáveis no modelo')
plt.show()


## 12. Conclusões

- A análise exploratória mostrou padrões relevantes de comportamento dos usuários ao longo do tempo.
- Variáveis temporais e operacionais ajudam a explicar parte do comportamento de cancelamento.
- O modelo inicial de Random Forest demonstrou potencial para apoiar previsões de cancelamento.
- Próximos passos incluem testar novos atributos, ajustar hiperparâmetros e comparar diferentes modelos.
